# FinSight AI — Research Notebook 03: Machine Learning Model Benchmarking
## Comparative Evaluation: Linear Regression vs. Random Forest vs. XGBoost

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "backend"))  # run from the ml_research/ folder
import numpy as np
import pandas as pd
from app.data import market_data as md
from app.ml.evaluation import evaluate_model
from app.ml.feature_engineering import FEATURE_COLUMNS, compute_features, create_target, time_series_split
from app.ml.models import LinearRegressionModel, RandomForestModel, XGBoostModel, model_availability

H = 5
raw, is_demo = md.history("NVDA", "5y", "1d")
feats = compute_features(raw)
X_all, y = create_target(feats, horizon=H)
X = X_all[FEATURE_COLUMNS]
close = X_all["Close"]
X_train, X_val, X_test, y_train, y_val, y_test = time_series_split(X, y, gap=H)
print(f"{len(X_train)} train / {len(X_val)} validation / {len(X_test)} test rows · simulated={is_demo}")

In [ ]:
# Train each model on returns, convert forecasts back to prices, evaluate on the unseen test set
models = {"Linear Regression": LinearRegressionModel, "Random Forest": RandomForestModel}
if model_availability()["xgboost"] is None:
    models["XGBoost"] = XGBoostModel

base = close.loc[X_test.index].values
actual = base * (1 + y_test.values)
results = {"Naive (no change)": {**evaluate_model(actual, base, "test", base), "directional_accuracy": None}}
for name, cls in models.items():
    m = cls("NVDA", horizon=H)
    m.fit(X_train, y_train, X_val, y_val)
    pred_ret = pd.Series(m.predict_full(X), index=X.index).loc[X_test.index].values
    results[name] = evaluate_model(actual, base * (1 + pred_ret), "test", base)

pd.DataFrame(results).T[["rmse", "mape", "directional_accuracy", "skill_vs_naive", "r_squared_returns"]].round(4)

**Reading the table:** `skill_vs_naive` > 0 means the model beats the random-walk forecast "tomorrow's price = today's price". For liquid stocks this is rarely more than a few percent — evidence for the weak-form efficient-market hypothesis.